# Revisão - Indústria Alimentícia

## 1. Importação das bibliotecas e definições gerais

In [15]:
import os
import pandas as pd
import numpy as np
import urllib3
import importlib
import functions_TratDados

importlib.reload(functions_TratDados)

# Desabilita avisos de requisições HTTPS sem verificação de certificado
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# Ignora especificamente o FutureWarning de "Downcasting" do pandas
warnings.filterwarnings(
    "ignore", 
    category=FutureWarning, 
    message="Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated"
)

import pandas as pd
import unicodedata # Serve para acessar e manipular as propriedades de caracteres

def clean_text(text):
    """
    Limpa uma string: remove acentos, espaços extras e converte para maiúsculo.

    Parâmetros:
        text (str): O texto a ser limpo.

    Retorna:
        str: O texto limpo, ou o valor original se for nulo.
    """
    # Se o valor for nulo (None, NaN), retorna como está para evitar erros.
    if pd.isna(text):
        return text

    # Garante que é string, remove espaços no início/fim e põe em maiúsculo.
    text = str(text).strip().upper()

    # Remove acentuação e caracteres especiais (ex: ç, á -> C, A).
    text = unicodedata.normalize('NFKD', text).encode('ASCII', 'ignore').decode('ASCII')

    return text

## 2. Criação da base geral

### 2.1. Funções

#### Função auxiliar para analisar CNPJ e CPF

In [16]:
def CNPJAnalysis(df, cnpj_column='mv.num_cpf_cnpj'):
    """
    Analyzes CNPJ/CPF data in a DataFrame, classifying documents and counting digit lengths.
    
    Parameters:
        df (pd.DataFrame): DataFrame containing the documents to analyze
        cnpj_column (str): Name of the column containing CPF/CNPJ numbers (default: 'mv.num_cpf_cnpj')
    
    Returns:
        dict: A dictionary with the count of documents by digit length
        pd.DataFrame: The original DataFrame with added 'tipo' column (CNPJ/CPF/outro)
    """
    
    # 1. Classify documents by length
    df['status_v01'] = np.where(
        df[cnpj_column].str.len() == 14, 'CNPJ - Considerado para análise',
        np.where(
            df[cnpj_column].str.len() == 11, 'CPF - Desconsiderado para análise',
            'CPF - Desconsiderado para análise'
        )
    )
    
    # 2. Initialize counter for digit lengths (4-14)
    contagem = {length: 0 for length in range(4, 15)}
    total_documents = len(df)
    
    # 3. Count documents by cleaned digit length
    for doc in df[cnpj_column]:
        # Remove all non-digit characters
        cleaned_doc = ''.join(filter(str.isdigit, str(doc)))
        length = len(cleaned_doc)
        
        if length in contagem:
            contagem[length] += 1
    
    # 4. Verification and reporting
    total_counted = sum(contagem.values())
    
    if total_counted == total_documents:
        print("✅ Todos os documentos foram contabilizados!")
    else:
        print(f"⚠️ Atenção: {total_documents - total_counted} documentos não se enquadram nos tamanhos 4-14 dígitos")
    
    # 5. Detailed report
    print("\nDistribuição de dígitos:")
    for length, quantity in sorted(contagem.items()):
        if quantity > 0:  # Only show lengths that actually appear
            print(f'{quantity:>5} documentos com {length:>2} dígitos ({quantity/total_documents:.1%})')
    
    print(f"\nTotal contado: {total_counted} de {total_documents} documentos\nApenas serão considerados documentos com 14 dígitos (CNPJs)")
    
    return



#### Funções de download da base de dados online com CNPJ e localização


In [17]:
def download_ibama_ctf_data(repo_path):
    '''
    Automatiza o processo de download, processamento e consolidação de dados de
    Pessoas Jurídicas do Cadastro Técnico Federal (CTF) do IBAMA.

    O fluxo de trabalho da função é o seguinte:
    1. Cria as estruturas de pastas necessárias ('inputs/dadosIbamaCNPJ_Download'
       para arquivos brutos e 'inputs/dadosIbamaCNPJ_Consolidado' para o final).
    2. Pergunta ao usuário se os arquivos já foram baixados para pular a etapa de download.
    3. Se o download for necessário, itera por todas as Unidades da Federação (UFs),
       baixa o arquivo CSV correspondente e o salva na pasta de dados brutos.
    4. Cada arquivo baixado é processado individualmente para padronizar CNPJs e
       nomes de colunas.
    5. Ao final, todos os dados processados são consolidados em um único DataFrame.
    6. Colunas de texto ('Municipio', 'Razao Social') passam por uma limpeza final.
    7. O DataFrame consolidado é salvo como um único arquivo CSV na pasta de
       dados processados.

    Parâmetros:
        repo_path (str): Caminho para a pasta raiz do projeto. As subpastas de
                         dados serão criadas dentro deste caminho.

    Retorna:
        pandas.DataFrame: Um DataFrame contendo todos os dados consolidados e limpos.
        None: Retorna None se o processo falhar ou se nenhum arquivo for processado.
    '''
    
    # Define os caminhos para as pastas de dados brutos e processados
    raw_dir = os.path.join(repo_path, 'inputs', 'MaterialBaixado','PJ_UF')
    processed_dir = os.path.join(repo_path, 'inputs', 'MaterialBaixado')
    
    # Garante que as pastas de destino existam; se não, elas são criadas
    os.makedirs(raw_dir, exist_ok=True)
    os.makedirs(processed_dir, exist_ok=True)
    
    # Interage com o usuário para saber se a etapa de download pode ser pulada
    start = input('Você já tem os arquivos baixados? (s/n) ')
    
    # Se o usuário responder 's', o script carrega o arquivo já consolidado
    if start.lower() == 's':
        
        # Caminho para o arquivo final que deveria existir
        final_file_path = os.path.join(processed_dir, "PJ_BR.csv")
        print(f"Carregando arquivo consolidado de: {final_file_path}")
        
        # Lê o CSV consolidado, garantindo a tipagem correta de colunas-chave
        df_final = pd.read_csv(final_file_path,
                               dtype={'CNPJ': str,
                                      'Codigo da atividade': str,
                                      'Codigo da categoria': str},
                               keep_default_na=False)
               
        return df_final
    
    else:
        # Bloco de execução para o download e processamento dos dados
        print("Iniciando o download e processamento dos dados...")
        base_url = "http://dadosabertos.ibama.gov.br/dados/CTF/APP/"
        
        # Lista de todas as Unidades da Federação do Brasil
        ufs = [
            'AC', 'AL', 'AP', 'AM', 'BA', 'CE', 'DF', 'ES', 'GO',
            'MA', 'MT', 'MS', 'MG', 'PA', 'PB', 'PR', 'PE', 'PI',
            'RJ', 'RN', 'RS', 'RO', 'RR', 'SC', 'SP', 'SE', 'TO'
        ]
        
        # Lista vazia para armazenar os DataFrames de cada estado após o processamento
        pessoas_juridicas_br = []
        
        # Itera sobre cada UF para baixar e processar os dados
        for uf in ufs:
            try:
                # Constrói a URL completa para o arquivo CSV do estado atual
                url = f"{base_url}{uf}/pessoasJuridicas.csv"
                print(f"Baixando {uf}...")
                
                # Faz a requisição GET para baixar o arquivo
                # timeout=30: define um tempo limite de 30 segundos
                # verify=False: ignora erros de certificado SSL (usar com cautela)
                response = requests.get(url, verify=False, timeout=30) 
                
                # Verifica se a requisição foi bem-sucedida (status code 2xx)
                response.raise_for_status() 
                
                # Define o caminho completo para salvar o arquivo bruto baixado
                output_path = os.path.join(raw_dir, f"pessoasJuridicas_{uf}.csv")
                
                # Abre o arquivo em modo de escrita binária ('wb') e salva o conteúdo
                with open(output_path, 'wb') as f:
                    f.write(response.content)
                
                # Inicia o processamento do arquivo recém-baixado
                try:
                    # Carrega o arquivo CSV em um DataFrame, especificando o delimitador e os tipos de dados
                    df = pd.read_csv(output_path,
                                     delimiter=';',
                                     dtype={'CNPJ': str,
                                            'Codigo da atividade': str,
                                            'Codigo da categoria': str},
                                     keep_default_na=False)
                    
                    # Padroniza a coluna de CNPJ para ter sempre 14 dígitos, com zeros à esquerda
                    df['CNPJ'] = df['CNPJ'].str.zfill(14) 
                    
                    # Limpa e padroniza os nomes das colunas usando a função externa clean_text
                    df.columns = df.columns.map(clean_text)
                    
                    # Adiciona o DataFrame do estado, já limpo, à lista de consolidação
                    pessoas_juridicas_br.append(df)
                    
                # Captura e informa erros que possam ocorrer durante o processamento do arquivo
                except Exception as e:
                    print(f"Erro ao processar {uf}: {e}")
                    
            # Captura e informa erros relacionados ao download (ex: rede, URL inválida)
            except requests.exceptions.RequestException as e:
                print(f"Erro ao baixar {uf}: {e}")
        
        # Após o loop, verifica se algum dado foi processado antes de consolidar
        if pessoas_juridicas_br:
            print("\nConsolidando todos os dados...")
            # Junta todos os DataFrames da lista em um único DataFrame
            df_final = pd.concat(pessoas_juridicas_br, ignore_index=True)
            
            # Aplica a função de limpeza nas células das colunas 'Municipio' e 'Razao Social'
            df_final['MUNICIPIO'] = df_final['MUNICIPIO'].apply(clean_text)
            df_final['RAZAO SOCIAL'] = df_final['RAZAO SOCIAL'].apply(clean_text)
            df_final['MUNICIPIO'] = df_final['MUNICIPIO'].str.replace(
                r"TRAJANO DE MORAIS", "TRAJANO DE MORAES", regex=True
                )
            
            # Extrair os anos de inicio e fim das datas
            # ANO_INICIO
            df_final['ANO_INICIO'] = (
                df_final['DATA DE INICIO DA ATIVIDADE']
                .fillna('0000')                    # trata valores nulos
                .replace('', '0000')               # trata strings vazias
                .str[-4:]                          # pega os 4 últimos caracteres
                .astype(int)                       # converte para inteiro
            )
            
            # ANO_FIM
            df_final['ANO_FIM'] = (
                df_final['DATA DE TERMINO DA ATIVIDADE']
                .fillna('0000')                    # trata valores nulos
                .replace('', '0000')               # trata strings vazias
                .str[-4:]
                .astype(int)
            )
            #Onde a data de inicio e fim de atividade for vazio, inserir np.nan
   
            # Define o caminho e nome do arquivo final e o salva em formato CSV
            output_file = os.path.join(processed_dir, "PJ_BR.csv")
            df_final.to_csv(output_file, index=False, encoding='utf-8')
            print("Dados baixados e processados com sucesso!")
            
            # Retorna o DataFrame final
            return df_final
            
    # Se a lista 'pessoas_juridicas_br' estiver vazia, retorna None
    print("Nenhum dado foi processado.")
    return None

#### Funções de importação de base de dados confidencial encaminhada pelo IBAMA

Processa dados de produção do IBAMA a partir de arquivos XLSX, limpando e consolidando os dados. Retorna:
- pd.DataFrame: DataFrame com os dados processados
- str: Caminho do arquivo CSV salvo

In [18]:
def ibama_production_data_17_23(repo_path):
    """
    """

    # Define os caminhos para as pastas de dados brutos e processados
    raw_dir = os.path.join(repo_path, 'inputs','DadosProduçãoIndustrial')
    processed_dir = os.path.join(repo_path, 'inputs','DadosProduçãoIndustrial')
    
    # Garante que as pastas de destino existam; se não, elas são criadas
    os.makedirs(raw_dir, exist_ok=True)
    os.makedirs(processed_dir, exist_ok=True)
    
    try:
        # Caminho completo do arquivo
        file_path = os.path.join(raw_dir, 'DadosProduçãoBruto.xlsx')
        
        # Lê as duas abas do Excel
        aba1 = pd.read_excel(file_path, sheet_name=0, dtype={'mv.num_cpf_cnpj': str}) # 0 para a primeira aba
        aba2 = pd.read_excel(file_path, sheet_name=1, dtype={'mv.num_cpf_cnpj': str}) # 1 para segunda aba
                
        # Combina as abas, deleta as linhas duplicadas
        df_ibama_prod = pd.concat([aba1, aba2], ignore_index=True)
        
        
        # Padronização dos dados com a função clean_text
        df_ibama_clean = df_ibama_prod.copy()
        df_ibama_clean['mv.nom_municipio'] = df_ibama_clean['mv.nom_municipio'].apply(clean_text)
        df_ibama_clean['mv.nom_pessoa'] = df_ibama_clean['mv.nom_pessoa'].apply(clean_text)
        df_ibama_clean['mv.nom_municipio'] = df_ibama_clean['mv.nom_municipio'].str.replace(
            r"SANT'? ?ANA DO LIVRAMENTO", "SANTANA DO LIVRAMENTO", regex=True
            )
        df_ibama_clean['mv.nom_municipio'] = df_ibama_clean['mv.nom_municipio'].str.replace(
            r"PRESIDENTE CASTELLO BRANCO", "PRESIDENTE CASTELO BRANCO", regex=True
            )
              
        # Salva o resultado
        output_file = os.path.join(processed_dir, 'DadosProduçãoTratado_17_23.csv')
        df_ibama_clean.to_csv(output_file, index=False, encoding='utf-8') #Remove índices
        
        return df_ibama_clean
        
    # Caso dê erro, informa ao usuário
    except FileNotFoundError:
        raise FileNotFoundError(f"Arquivo RAPP.xlsx não encontrado em {raw_dir}")
    except Exception as e:
        raise Exception(f"Erro ao processar dados: {str(e)}")

#### Funções que mescla produção com coordenadas

In [19]:
def merge_cnpj_prod(cnpj,prod):
    '''
    
    Processa dados de atividade por CNPJ e Município para obter coordenada,
    código de atividade e código de categoria; Remove os que são CPF   
    
    Parâmetros: 
        - cnpj, prod = Base de dados de CNPJ e de Produção do ibama, e baixado pelas funções:
            -download_ibama_ctf_data
            -ibama_production_data
            
     Retorna:
         pd.DataFrame: DataFrame com os dados concatenados
         
    '''
    
    # Uma mesma indústria com o mesmo cnpj e lat lon produz produtos diferentes
    # Ou seja, multiplos codigo de categoria e codigo de atividade
    # Por isso, fiz um groupby onde os códigos de categoria e atividade viram uma lista para um mesmo CNPJ
    # acabei não utilizando estes códigos, mas mantive por segurança
    cnpj = cnpj.groupby(['CNPJ', 'MUNICIPIO', 'LATITUDE', 'LONGITUDE','ESTADO','SITUACAO CADASTRAL']).agg({
        'CODIGO DA CATEGORIA': list,
        'CODIGO DA ATIVIDADE': list,
        'ANO_INICIO': list,
        'ANO_FIM': list
    }).reset_index()
    
    
    # Fazer o merge com o df_ibama (sem duplicar linhas)
    df_ibama_completo = pd.merge(
        left=cnpj, #tabela unida a esqueda
        right=prod, #tabela unida a direita
        left_on=['CNPJ', 'MUNICIPIO'], #chaves da tabela a esqueda
        right_on=['mv.num_cpf_cnpj', 'mv.nom_municipio'], #chaves da tabela a direita
        how='right', # todas as linhas da tabela da direita (prod) serão mantidas.
        indicator=True #informa a condição da mesclagem (right_only - não estava em CNPJ)
    )
    
       
    return df_ibama_completo

#### Função que agrupa e agrega os valores

In [20]:
def agrupar_e_somar_dados(df: pd.DataFrame) -> pd.DataFrame:
    """
    Agrupa um DataFrame por um conjunto de colunas-chave e agrega os valores.

    As colunas numéricas (não presentes nas chaves de agrupamento) são somadas.
    As colunas não numéricas são preenchidas com o primeiro valor do grupo.

    Args:
        df (pd.DataFrame): O DataFrame de entrada para ser processado.

    Returns:
        pd.DataFrame: Um novo DataFrame com os dados agrupados e agregados.
    """
    # 1. Define as colunas que serão a "chave" para o agrupamento
    colunas_agrupamento = [
        'mv.num_cpf_cnpj', 
        'mv.nom_pessoa',
        'mv.nom_municipio',
        'num_ano',
        'cod_produto',
        'unidade_medida',
        'sig_unidmed'
    ]

    print(f"Número de linhas antes do agrupamento: {len(df)}")

    # 2. Cria dinamicamente as regras de agregação
    # O objetivo é dizer ao pandas o que fazer com cada coluna que NÃO está no agrupamento.
    regras_agregacao = {}
    colunas_para_agregar = [col for col in df.columns if col not in colunas_agrupamento]

    for coluna in colunas_para_agregar:
        # Verifica se a coluna contém dados numéricos (int, float, etc.)
        if pd.api.types.is_numeric_dtype(df[coluna]):
            regras_agregacao[coluna] = 'sum'  # Se for numérica, some
        else:
            regras_agregacao[coluna] = 'first' # Se não for, pegue o primeiro valor

    # 3. Executa o agrupamento e a agregação
    # .groupby() cria os grupos
    # .agg() aplica as regras que definimos
    # .reset_index() transforma as colunas de agrupamento de volta em colunas normais
    df_agrupado = df.groupby(colunas_agrupamento).agg(regras_agregacao).reset_index()

    print(f"Número de linhas após o agrupamento: {len(df_agrupado)}")
    
    return df_agrupado

### 2.2. Código

In [ ]:
# Caminho da pasta do projeto
# Dentro do repo_path deve-se ter as pastas
# repo_path
#   |--figures
#   |--inputs
#   |--outputs
#       |--log
#   |--scripts

repo_path = os.path.dirname(os.getcwd())

# Faz o downloado da base de dados com CNPJ + Coordenadas
df_ibama_cnpj = download_ibama_ctf_data(repo_path)

# Importa DF com Dados de produção com CNPJ + Código de Produto + Produção
df_ibama_prod = ibama_production_data_17_23(repo_path)

#Garante que não existam linhas duplicadas com base em: 'mv.num_cpf_cnpj', 'mv.nom_pessoa',
#'mv.nom_municipio','num_ano','cod_produto','unidade_medida','sig_unidmed'
df_ibama_prod = agrupar_e_somar_dados(df_ibama_prod)

# DF com Produção + Código de produto + Coordenadas
#mesclar p obter coordenadas e código de atividade
df_ibama = merge_cnpj_prod(df_ibama_cnpj,df_ibama_prod) 

#contabiliza quantos são CPF (11 dígitos) e quantos são CNPJ (14 dígitos)
CNPJAnalysis(df_ibama)

#Exporta log com dados desconsiderados
df_ibama.to_excel(os.path.join(repo_path,'outputs','log','log_v01_remocaoCPF.xlsx'))

#Remover CPFs (desconsiderável)
df_ibama = df_ibama[df_ibama['mv.num_cpf_cnpj'].str.len() == 14]

Carregando arquivo consolidado de: c:\Users\glima\OneDrive\Documentos\Mestrado_GitHub\006.2026 - Revisão TCC\inputs\MaterialBaixado\PJ_BR.csv


## 3. Base com NFR + Código de Produto + Produção

### 3.1. Funções

Importa e limpa a tabela de códigos de produtos do IBGE (PRODLIST) APENAS PRODUTOS ALIMENTÍCIOS (escopo inicial do TCC).
- Verificar online: https://servicos.ibama.gov.br/ctfcd/manual/html/lista_produtos.htm
- Baixar excel: https://www.ibge.gov.br/estatisticas/metodos-e-classificacoes/classificacoes-e-listas-estatisticas/9153-lista-de-produtos-da-industria.html
   
A função lê um arquivo Excel, remove cabeçalhos repetidos e linhas indesejadas, e retorna um DataFrame pronto para uso. Retorna:
- pd.DataFrame: Tabela de códigos de produtos limpa.  

In [ ]:
def import_treat_export_food_code(repo_path):
    raw_dir = os.path.join(repo_path, 'inputs','MaterialBaixado')
    processed_dir = os.path.join(repo_path, 'outputs')
    
    # Garante que as pastas de destino existam; se não, elas são criadas
    os.makedirs(raw_dir, exist_ok=True)
    os.makedirs(processed_dir, exist_ok=True)
    
    # Caminho do arquivo
    xlsx_path = os.path.join(raw_dir,'CódigosProdutosIBGE.xlsx')

    # Lê o CSV com header na linha 1 (índice 1)
    cod_produto = pd.read_excel(xlsx_path, header=2, dtype={'PRODLIST': str})
    # Remove linhas com 'PRODLIST' ou NaN na coluna 'PRODLIST'
    cod_produto = cod_produto[~cod_produto['PRODLIST'].isin(['PRODLIST'])]
    cod_produto = cod_produto[~cod_produto['PRODLIST'].astype(str).str.startswith('CNAE')]
    cod_produto = cod_produto.dropna(subset=['PRODLIST'])

    # Reinicia o índice
    cod_produto.reset_index(drop=True, inplace=True)
       
    return cod_produto

Conecta a base de dados do IBAMA ao fator de emissão adotado manualmente

In [ ]:
def conecta_ibama_ef(df_ibama, df_ef, df_conector):
    # Garantir que os códigos estejam no mesmo tipo
    df_conector[['PRODLIST', 'NFR', 'Table']] = df_conector[['PRODLIST', 'NFR', 'Table']].astype(str)
    df_ibama['cod_produto'] = df_ibama['cod_produto'].astype(str)
    
    # Mesclar df_ibama com o conector via código do produto
    df_merged = df_ibama.merge(
        df_conector[['PRODLIST', 'NFR', 'Table']],
        left_on='cod_produto',
        right_on='PRODLIST',
        how='left'
    )

    # Realizar o merge com a base ef (EEA)
    df_final = df_merged.merge(
        df_ef,
        left_on=['NFR', 'Table'],
        right_on=['NFR', 'Table'],
        how='left'
    )

    # Remover coluna auxiliar
    #df_final = df_final.drop(columns=['PRODLIST'])

    return df_final

### 3.2. Código

Aqui eu importo a base de dados com os códigos de produto do IBGE, faço uma limpeza inicial e exporto para a pasta de outputs para classificar manualmente quais códigos se enquadram no item 2.h.2 do NFR (Alimentos e Bebidas)

In [ ]:
#Base de dados com todos os Códigos de Produto
cod_produto= import_treat_export_food_code(repo_path)

#vou filtrar os códigos de produto de interesse e exportar (itens que se enquadram no 2.h.2 no nfr)
cod_produto_interesse = cod_produto[
    cod_produto['PRODLIST'].astype(str).str.startswith(('10', '11'))
]

#exportei para classificar manualmente
cod_produto_interesse.to_excel(os.path.join(repo_path, 'outputs','CodProdutoParaClassificar.xlsx'), index=False)

Após classificar os códigos de produto em NFR, exportar manualmente para a pasta inputs. Os classificados como "Não" serão desconsiderados.

In [ ]:
#Importei material gerado manualmente
#VERIFICACAO
CodProdutoClassificadoNFR = pd.read_excel(os.path.join(repo_path,'inputs','MaterialGeradoManualmente','CodProdutoClassificadoNFR.xlsx'),
                                          dtype={'PRODLIST': str})

#filtrei os alimentos que tem emissões a serem consideradas
CodProdutoClassificadoNFR = CodProdutoClassificadoNFR[CodProdutoClassificadoNFR['EmissaoNMCOV_NFR'] != 'Não']

# Base de dados dos fatores de emissão tier 2
eea_ef = pd.read_csv(os.path.join(repo_path, 'inputs','MaterialBaixado', 'EF_tier2.csv'))

# Conexão das bases de dados de apenas os classificados como emissores de NMCOV
df_ibama_EF = conecta_ibama_ef(df_ibama,eea_ef,CodProdutoClassificadoNFR)

#log que indica os removidos por não serem alimentos emissores de COV
# 1. Crie a coluna com o valor padrão para todas as linhas
df_ibama_EF['status_v02'] = 'Dado filtrado'

# 2. Use .loc para encontrar as linhas que batem com a condição e mude o valor apenas nelas
df_ibama_EF.loc[df_ibama_EF['NFR'] == '2.H.2', 'status_v02'] = 'Alimento emissor de COV - Dado considerado'

#Exporta log com dados desconsiderados
df_ibama_EF.to_excel(os.path.join(repo_path,'outputs','log','log_v02_apenasAlimentosEmissoresMantidos.xlsx'))

#Filtrar apenas os produtos com emissão
df_ibama_EF = df_ibama_EF[df_ibama_EF['Table'].notna()]

## 4. Ajuste 01: Indústrias que reportaram mais de uma unidade de medida

### 4.1 Função

In [ ]:
# Classificar o tipo de bebida
def classificar_produto(codigo):
    if codigo.startswith('Sugar'):
        return 'Açucar','beige'
    elif codigo.startswith('Coffee'):
        return 'Torrefação do café','brown'
    elif codigo.startswith('Margarine'):
        return 'Margarina e gorduras sólidas','yellow'
    elif codigo.startswith('Cakes'):
        return 'Bolos, biscoitos e cereais matinais','grey'
    elif codigo.startswith(('Meat','Animal')):
        return 'Preparação de Carnes e Ração Animal','salmon'
    elif codigo.startswith('Wine'):
        return 'Vinho','purple'
    elif codigo.startswith(('White bread','Wholemeal')):
        return 'Pão','pink'
    elif codigo.startswith('Beer'):
        return 'Cerveja','goldenrod'
    else:
        return 'Destilados','lightblue'

### 4.2 Código

O objetivo desta seção é encontrar os grupos (CNPJ, Município, Produto) que possuem mais de uma unidade de medida declarada ao longo do tempo e corrigi-las para uma unidade coerente.

In [ ]:
# Para cada combinação única de [CNPJ, MUNICIPIO, cod_produto], calcula-se o total de registros.
# A função 'transform' aplica essa contagem a todas as linhas do grupo original,
# criando a coluna 'contagem_total_grupo' para uso posterior na análise.
df_ibama_EF['contagem_total_grupo'] = df_ibama_EF.groupby(
    ['CNPJ', 'MUNICIPIO', 'cod_produto']
)['cod_produto'].transform('count')

# Agrupa-se novamente e conta-se a frequência de cada 'unidade_medida' dentro do mesmo grupo.
# O resultado é um DataFrame ('df_valid_final') que mostra, por exemplo:
# CNPJ A, Produto X -> 'tonelada': 10 vezes, 'unidade': 2 vezes.
df_valid_final = df_ibama_EF.groupby(
    ['CNPJ', 'MUNICIPIO', 'cod_produto', 'contagem_total_grupo']
)['unidade_medida'].value_counts().reset_index(name='contagem_unidade')

# Um grupo é inconsistente se possuir mais de um tipo de unidade de medida.
# A lógica aqui é identificar no 'df_valid_final' os grupos que aparecem mais de uma vez.
grupos_inconsistentes = df_valid_final[df_valid_final.duplicated(
    subset=['CNPJ', 'MUNICIPIO', 'cod_produto'], keep=False
)]

# Gera uma lista limpa e sem duplicatas dos grupos que precisam ser corrigidos manualmente.
df_para_verificar = grupos_inconsistentes[['CNPJ', 'MUNICIPIO', 'cod_produto']] \
    .drop_duplicates() \
    .reset_index(drop=True)

# Exporta a lista de grupos inconsistentes para um arquivo Excel.
# Este arquivo servirá como um guia para a verificação manual.
df_para_verificar.to_excel(os.path.join(repo_path,
                                        'outputs',
                                        'Auxiliar_dfUnidadesInconsistentes_VerifManual.xlsx'),
                           index=False)


NameError: name 'df_ibama_EF' is not defined

Nesta etapa, o DataFrame principal é preparado e exportado para que as correções sejam feitas manualmente em um software de planilha. A correção manual consiste e inserir na coluna 'sig_unidmed_novo' a unidade adequada, de forma a deixar coerente com as demais.

In [ ]:
# Adiciona colunas para registrar as alterações, preservando os dados originais.
# 'status_v03': Descreve a ação tomada (ex: 'Unidade alterada: ...').
# 'sig_unidmed_novo' e 'qtd_produzida_novo': Receberão os valores corrigidos.
df_ibama_EF['status_v03'] = 'Unidade mantida conforme dado original'
df_ibama_EF['sig_unidmed_novo'] = df_ibama_EF['sig_unidmed']

# Exporta o DataFrame completo. O usuário deverá abrir este arquivo,
# corrigir os valores nas colunas '_novo' e justificar a alteração em 'status_v03'.
df_ibama_EF.to_excel(os.path.join(repo_path,
                                  'outputs',
                                  'vefirManual_01_unidadesPorProdutoPorProdutor.xlsx'))

Após a correção manual, o arquivo é reimportado para o script e uma cópia é salva como log para garantir a rastreabilidade do processo. Além disso, classificou-se os tipos de alimento por grande grupo.

In [ ]:
# Lê o arquivo Excel que foi modificado manualmente, contendo as unidades padronizadas.
# Este arquivo deve ser previamente salvo na pasta de inputs manuais.
df_ibama_EF_und = pd.read_excel(os.path.join(repo_path,
                                             'inputs',
                                             'MaterialGeradoManualmente',
                                             'vefirManual_01_unidadesPorProdutoPorProdutor.xlsx'),
                                dtype={'cod_produto': 'string','CNPJ':'string'})

# Salva uma cópia do DataFrame já corrigido no diretório de log.
# Esta ação preserva o estado dos dados após a correção manual
df_ibama_EF_und.to_excel(os.path.join(repo_path,
                                      'outputs',
                                      'log',
                                      'log_v03_AnaliseManualUndInconsistente.xlsx'))

#Agrupo em grupos de alimentos emissores de COV para análises posteriores
df_ibama_EF_und['tipo_industria_nfr'], df_ibama_EF_und['food_color'] = zip(
    *df_ibama_EF_und['Technology'].map(classificar_produto)
)

Será criado uma tabela única com todas as combinações de produto e unidade de medida existentes nos dados. Essa tabela será usada para realizar a correção das unidades.

In [ ]:
# Define as colunas que identificam uma combinação única de produto/unidade.
colunas_agrupamento = ['cod_produto', 'nom_produto', 'sig_unidmed_novo', 'Unit']

# Agrupa o DataFrame para encontrar todas as combinações únicas e conta
# quantas vezes cada uma aparece. A coluna 'contagem' serve como
# referência de frequência para a análise e preenchimento manual.
df_unidades_bruto = df_ibama_EF_und.groupby(colunas_agrupamento).size().reset_index(name='contagem')

# Renomeio as colunas para não ter duplicidade nos nomes posteriormente
df_unidades_bruto.rename(columns={"cod_produto": "cod_produto_fc",
                                  "nom_produto": "nom_produto_fc",
                                  "sig_unidmed_novo": "sig_unidmed_fc"})

# Exporta a tabela gerada para um arquivo Excel. Este arquivo deve ser
# preenchido manualmente com os fatores de conversão correspondentes.
df_unidades_bruto.to_excel(os.path.join(repo_path, 'outputs','UnidadesFatorConversãoBruto.xlsx'), index=False)

Após o preenchimento manual, o arquivo com os fatores de conversão é importado das pasta de inputs manuais para ser mesclado com a base de dados principal.

In [ ]:
# Carrega a planilha Excel que agora contém os fatores de conversão preenchidos.
# 'cod_produto_fc' é lido como string para evitar problemas de formatação.
df_unidades = pd.read_excel(os.path.join(repo_path, 'inputs',
                                         'MaterialGeradoManualmente',
                                         'UnidadesFatorConversão.xlsx'),
                            dtype={'cod_produto_fc': str})

Mesclo o DataFrame de produção ('df_ibama_EF_und') com os fatores de conversão ('df_unidades') para que cada linha de produção tenha seu respectivo fator para a padronização da unidade.

In [ ]:
# Realiza a junção (merge) dos DataFrames.
# As chaves 'left_on' e 'right_on' conectam o produto e a unidade de cada tabela.
# O 'how='left'' garante que todos os registros de produção originais sejam mantidos
df_producao_bruto = pd.merge(
        left=df_ibama_EF_und,
        right=df_unidades,
        left_on=['cod_produto', 'sig_unidmed_novo'],
        right_on=['cod_produto_fc', 'sig_unidmed_fc'],
        how='left')

# Multiplica a quantidade produzida original pelo seu fator de conversão correspondente.
df_producao_bruto['prodtonhl_v0'] = (df_producao_bruto['qtd_produzida'].astype(float) * df_producao_bruto['fatorConversao'].astype(float))

## 5. Análise 02: Alteração da escala de produção

### 5.1 Código

Nessa etapa, buscou-se a correção de escalas de produção, dividindo ou multiplicando o valor de produção por múltimos de 10, tornando os valores coerentes. Esta etapa foi realizada concomitantemente com a exportação da comparação com o PIA, buscando-se dados coerentes com a base de dados indicadora.

Dessa forma, primeiro exportou-se uma base de dados com a produção classificada por ser maior ou menor que o Q90 do grupo, e vou analisar os CNPJs classificados como sim. Em seguida, vou indicadar a escala em que deve-se aumentar ou diminuir o valor para ele se tornar coerente com a escala de produção

In [ ]:
# 1. Definir as colunas de agrupamento e a coluna de valor
group_cols = 'tipo_industria_nfr'
value_col = 'prodtonhl_v0'
percentil = 0.90

# 2. Calcular o valor do percentil 95 para cada grupo
# Usamos transform() para que o resultado tenha o mesmo índice do df original
p90_grupo = df_producao_bruto.groupby(group_cols)[value_col].transform(lambda x: x.quantile(percentil))

# 3. Adicionar o valor do p95 ao DataFrame para facilitar a análise no Excel
df_producao_bruto['p90_do_grupo_v0'] = p90_grupo

# 4. Filtrar as linhas onde a produção é maior que o p95 do seu grupo
mascara_outliers_p90 = (df_producao_bruto[value_col] > df_producao_bruto['p90_do_grupo_v0'])

df_producao_bruto['status_v04'] = np.where(mascara_outliers_p90==True,
                                             'produção_superior_q90_v0',
                                             'produção_inferior_q90_v0'
                                             )

df_producao_bruto['status_v05'] = 'Fator de Escala Mantido'
df_producao_bruto['fator_escala'] = 1
df_producao_bruto['prod_temp_testes'] = df_producao_bruto['prodtonhl_v0']

Caso não seja coerente, vou indicar "Aplicação de favor de escala, e colocar um valor múltiplo de 10 para multiplicar (pode ser 1/10 e afins) caso valor sejam suspeito de não ser produção ou incoerente, pode-se zerar este fator de escala.

In [ ]:
# Exportar df_prod_notnull como inventário_intermed para verif manual 2
df_producao_bruto.to_excel(os.path.join(repo_path,'outputs','vefirManual_02_fatorEscala.xlsx'), index=False)

df_outliers_para_verificar =df_producao_bruto['mv.num_cpf_cnpj'][mascara_outliers_p90].copy().drop_duplicates()

# Exportar o resultado para um arquivo Excel
df_outliers_para_verificar.to_excel(os.path.join(repo_path,'outputs','Auxiliar_fatorEscala.xlsx'), index=False)

# Importar valores analisados
df_producao_escala = pd.read_excel(os.path.join(repo_path,'inputs',
                                                'MaterialGeradoManualmente','vefirManual_02_fatorEscala.xlsx'),
                                   dtype={'cod_produto':'string','CNPJ':'string'})

df_producao_escala = df_producao_escala.drop(['prod_temp_testes'], axis=1)

df_producao_escala['CNPJ'] = (
    df_producao_escala['CNPJ']
    .astype(str)
    .str.strip()
    .str.rjust(14, '0')
)

#Calcula nova versão de prodtonhl_v0 --> v1
df_producao_escala['prodtonhl_v1'] = df_producao_escala['prodtonhl_v0'] * df_producao_escala['fator_escala']

## 6. Análise 03: Remoção de outliers, filtro e preenchimento de série

### 6.1 Funções

Função de tratamento de outliers utilizada

In [ ]:
def tratamento_outliers_v3(
    df: pd.DataFrame,
    janela_movel: int = 5,
    fator_mediana: float = 3.0,
    fator_aumento_anual: float = 2.0,
    fator_reducao_anual: float = 0.5
) -> pd.DataFrame:
    """
    Realiza um tratamento completo e rastreável de séries temporais de produção.

    O processo é dividido em etapas sequenciais, onde cada etapa principal gera uma
    nova coluna de produção (`prodtonhl_vX`) e uma coluna de status (`status_v0X`),
    permitindo uma auditoria clara das transformações aplicadas a cada registro.

    Etapas:
    1.  **Preparação (v1):**
        - `prodtonhl_v1`: Coluna de produção original, após consolidação de duplicatas.
    2.  **Filtragem (v2):**
        - `prodtonhl_v2`: Resultado da aplicação de filtros. Séries com histórico
          insuficiente ou inteiramente zeradas são marcadas e seus valores zerados
          nesta coluna, sendo excluídas das etapas seguintes.
        - `status_v06`: Descreve o resultado da Etapa 1 (ex: 'Apto para Análise',
          'Histórico Insuficiente', 'Série Zerada').
    3.  **Correção de Outliers (v3):**
        - `prodtonhl_v3`: Resultado da correção de outliers identificados em `v2`.
          Os outliers são substituídos por valores interpolados/extrapolados.
        - `status_v07`: Descreve a correção aplicada (ex: 'Valor Mantido',
          'Corrigido (Interpolação)').
    4.  **Preenchimento de Lacunas (v4):**
        - `prodtonhl_v4`: Versão final da série temporal, com anos faltantes
          preenchidos com base no status cadastral e na densidade de dados.
        - `status_v08`: Descreve o preenchimento (ex: 'Valor Mantido',
          'Preenchido (Global - Interpolação)').

    Args:
        df: DataFrame contendo os dados de produção. Deve incluir colunas de
            agrupamento, ano, valor de produção e situação cadastral.
        janela_movel: Tamanho da janela para cálculo da mediana móvel (detecção de outliers).
        fator_mediana: Fator multiplicativo para definir os limites da mediana móvel.
        fator_aumento_anual: Fator máximo de aumento anual permitido.
        fator_reducao_anual: Fator máximo de redução anual permitido.

    Returns:
        Um DataFrame com as colunas originais e as novas colunas de produção e status,
        refletindo cada etapa do tratamento.
    """
    print("Iniciando o tratamento de dados v4 (Etapas em Colunas)...")

    # --- 0. PREPARAÇÃO E VALIDAÇÃO ---
    df_processado = df.copy()
    group_cols = ['CNPJ', 'MUNICIPIO', 'cod_produto']
    year_col = 'num_ano'
    
    colunas_essenciais = group_cols + [year_col, 'prodtonhl_v1', 'SITUACAO CADASTRAL']
    for col in colunas_essenciais:
        if col not in df_processado.columns:
            raise ValueError(f"A coluna necessária '{col}' não foi encontrada no DataFrame.")

    agg_cols = group_cols + [year_col]
    if df_processado.duplicated(subset=agg_cols).any():
        print("-> Consolidando registros duplicados (usando 'mean')...")
        agg_dict = {'prodtonhl_v1': 'mean'}
        other_cols = {c: 'first' for c in df.columns if c not in agg_cols and c != 'prodtonhl_v1'}
        agg_dict.update(other_cols)
        df_processado = df_processado.groupby(agg_cols, as_index=False).agg(agg_dict)

    # --- [ADIÇÃO] Armazena a contagem total de linhas após a consolidação ---
    total_inicial_consolidado = len(df_processado)
    print(f"-> {total_inicial_consolidado} registros únicos (consolidados) para processar.")
    # --- [FIM ADIÇÃO] ---

    df_processado['prodtonhl_v2'] = df_processado['prodtonhl_v1']
    df_processado['status_v06'] = 'Apto para Análise'
    df_processado['status_v07'] = ''
    df_processado['status_v08'] = ''

    # --- 1. FILTRAGEM (Cria prodtonhl_v2 e status_v06) ---
    print("Etapa 1: Aplicando filtros (cria prodtonhl_v2)...")

    def _verificar_historico_suficiente(grupo: pd.DataFrame) -> bool:
        anos = sorted(grupo[year_col].unique())
        if len(anos) >= 5: return True
        if len(anos) >= 3:
            for i in range(len(anos) - 2):
                if anos[i+1] - anos[i] == 1 and anos[i+2] - anos[i+1] == 1:
                    return True
        return False

    ids_suficientes = df_processado.groupby(group_cols).filter(_verificar_historico_suficiente)
    mascara_insuficiente = ~df_processado.index.isin(ids_suficientes.index)
    
    df_processado.loc[mascara_insuficiente, 'status_v06'] = 'Histórico Insuficiente'
    df_processado.loc[mascara_insuficiente, 'prodtonhl_v2'] = 0
    
    df_filtrado = df_processado[~mascara_insuficiente].copy()
    
    if not df_filtrado.empty:
        somas_grupo = df_filtrado.groupby(group_cols)['prodtonhl_v1'].transform('sum')
        mascara_zerada = somas_grupo == 0
        indices_zerados = df_filtrado[mascara_zerada].index
        
        df_processado.loc[indices_zerados, 'status_v06'] = 'Série Zerada'
        df_processado.loc[indices_zerados, 'prodtonhl_v2'] = 0
        
        df_filtrado = df_filtrado[~mascara_zerada]

    # --- [RESUMO ETAPA 1] ---
    try:
        total_s1 = len(df_processado) # É igual a total_inicial_consolidado
        aptos_s1 = (df_processado['status_v06'] == 'Apto para Análise').sum()
        filtrados_s1 = total_s1 - aptos_s1
        perc_aptos = (aptos_s1 / total_s1) * 100 if total_s1 > 0 else 0
        
        print("\n--- Resumo Etapa 1 (Filtragem) ---")
        print(f"Total de linhas processadas: {total_s1}")
        print(f"Linhas Aptas p/ Análise:     {aptos_s1} ({perc_aptos:.2f}%)")
        print(f"Linhas Filtradas:            {filtrados_s1}")
        print(f"  - Histórico Insuficiente:  {(df_processado['status_v06'] == 'Histórico Insuficiente').sum()}")
        print(f"  - Série Zerada:            {(df_processado['status_v06'] == 'Série Zerada').sum()}")
        print("-----------------------------------\n")
    except Exception as e:
        print(f"*** Erro ao gerar resumo S1: {e} ***")
    # --- [FIM RESUMO ETAPA 1] ---
    
    # --- ETAPA 2: CORREÇÃO DE OUTLIERS (Cria prodtonhl_v3 e status_v07) ---
    print("Etapa 2: Corrigindo outliers (cria prodtonhl_v3)...")
    
    df_processado['status_v07'] = 'Não Aplicável'
    df_processado['prodtonhl_v3'] = df_processado['prodtonhl_v2']

    if not df_filtrado.empty:
        df_filtrado = df_filtrado.sort_values(by=agg_cols)
        
        medianas_moveis = df_filtrado.groupby(group_cols)['prodtonhl_v2'].transform(lambda x: x.rolling(window=janela_movel, min_periods=1, center=True).median())
        lim_sup, lim_inf = medianas_moveis * fator_mediana, medianas_moveis / fator_mediana
        m_mediana = ((df_filtrado['prodtonhl_v2'] > lim_sup) | (df_filtrado['prodtonhl_v2'] < lim_inf)) & (medianas_moveis > 0)
        
        prod_ant = df_filtrado.groupby(group_cols)['prodtonhl_v2'].shift(1).replace(0, np.nan)
        razao = df_filtrado['prodtonhl_v2'] / prod_ant
        m_anual = (razao > fator_aumento_anual) | (razao < fator_reducao_anual)
        
        df_filtrado['flag_outlier'] = m_mediana | m_anual
        print(f"-> {df_filtrado['flag_outlier'].sum()} outliers sinalizados para correção.")
        
        df_filtrado['status_v07'] = 'Valor Mantido'

        def _corrigir_outliers(grupo):
            pontos_validos = grupo[~grupo['flag_outlier']]
            for idx in grupo[grupo['flag_outlier']].index:
                ano_outlier = grupo.loc[idx, year_col]
                validos_ant = pontos_validos[pontos_validos[year_col] < ano_outlier]
                validos_pos = pontos_validos[pontos_validos[year_col] > ano_outlier]
                valor_sub, metodo = np.nan, ""

                if not validos_ant.empty and not validos_pos.empty:
                    p_ant, p_pos = validos_ant.iloc[-1], validos_pos.iloc[0]
                    dy, dx = p_pos['prodtonhl_v2'] - p_ant['prodtonhl_v2'], p_pos[year_col] - p_ant[year_col]
                    if dx > 0: valor_sub, metodo = p_ant['prodtonhl_v2'] + dy * ((ano_outlier - p_ant[year_col]) / dx), "Corrigido (Interpolação)"
                elif len(validos_ant) >= 2:
                    p1, p2 = validos_ant.iloc[-1], validos_ant.iloc[-2]
                    dy, dx = p1['prodtonhl_v2'] - p2['prodtonhl_v2'], p1[year_col] - p2[year_col]
                    if dx > 0: valor_sub, metodo = p1['prodtonhl_v2'] + (dy/dx) * (ano_outlier - p1[year_col]), "Corrigido (Extrapolação Fwd)"
                elif len(validos_pos) >= 2:
                    p1, p2 = validos_pos.iloc[0], validos_pos.iloc[1]
                    dy, dx = p2['prodtonhl_v2'] - p1['prodtonhl_v2'], p2[year_col] - p1[year_col]
                    if dx > 0: valor_sub, metodo = p1['prodtonhl_v2'] + (dy/dx) * (ano_outlier - p1[year_col]), "Corrigido (Extrapolação Bwd)"
                else:
                    mediana = pontos_validos['prodtonhl_v2'].median()
                    if pd.notna(mediana): valor_sub, metodo = mediana, "Corrigido (Mediana Série - Fallback)"

                if pd.notna(valor_sub):
                    grupo.loc[idx, 'prodtonhl_v3'] = max(0, valor_sub)
                    grupo.loc[idx, 'status_v07'] = metodo
            return grupo

        df_corrigido = df_filtrado.groupby(group_cols, group_keys=False).apply(_corrigir_outliers, include_groups=False)
        df_processado.update(df_corrigido)

    # --- [RESUMO ETAPA 2] ---
    try:
        # Base de análise são os aptos da S1
        base_s2 = (df_processado['status_v06'] == 'Apto para Análise').sum() 
        if base_s2 > 0:
            mantidos_s2 = (df_processado['status_v07'] == 'Valor Mantido').sum()
            # Conta qualquer status que comece com 'Corrigido'
            corrigidos_s2 = (df_processado['status_v07'].str.startswith('Corrigido', na=False)).sum()
            perc_corrigidos = (corrigidos_s2 / base_s2) * 100
            
            print("\n--- Resumo Etapa 2 (Outliers) ---")
            print(f"Total de linhas analisadas:     {base_s2} (Aptos da Etapa 1)")
            print(f"Linhas com Valor Mantido:       {mantidos_s2}")
            print(f"Linhas Corrigidas (Outliers): {corrigidos_s2} ({perc_corrigidos:.2f}%)")
            print("--------------------------------------\n")
        else:
             print("\n--- Resumo Etapa 2 (Outliers) ---")
             print("Nenhuma linha apta para análise de outliers.")
             print("--------------------------------------\n")
    except Exception as e:
        print(f"*** Erro ao gerar resumo S2: {e} ***")
    # --- [FIM RESUMO ETAPA 2] ---
    
    # --- ETAPA 3: PREENCHIMENTO DE LACUNAS (Cria prodtonhl_v4 e status_v08) ---
    print("Etapa 3: Preenchendo lacunas (cria prodtonhl_v4)...")
    
    df_processado['status_v08'] = 'Não Aplicável'
    df_processado['prodtonhl_v4'] = df_processado['prodtonhl_v3']
    
    ano_min_geral, ano_max_geral = df_processado[year_col].min(), df_processado[year_col].max()
    
    def _preencher_grupo(grupo, ano_min_g, ano_max_g):
        # --- SETUP INICIAL ---
        status_cadastral = str(grupo['SITUACAO CADASTRAL'].iloc[0])
        num_pontos = grupo.shape[0]
        densidade = num_pontos / (ano_max_g - ano_min_g + 1)
        
        grupo['status_v08'] = 'Valor Mantido'
    
        # --- LÓGICA DE DECISÃO DE PREENCHIMENTO ---
        if status_cadastral.upper() == 'ATIVA' and densidade >= 0.75:
            intervalo = range(ano_min_g, ano_max_g + 1)
            status_preenchimento = 'Preenchido (Global - Interpolação)'
    
        elif status_cadastral.upper().startswith('ENCERRAD') or (status_cadastral.upper() == 'ATIVA' and densidade < 0.75):
            intervalo = range(grupo['num_ano'].min(), grupo['num_ano'].max() + 1)
            status_preenchimento = 'Preenchido (Local - Interpolação)'
        
        else:
            grupo['prodtonhl_v4'] = 0
            grupo['status_v08'] = 'Zerado (Status Inválido/Outro)'
            return grupo
    
        # --- EXECUÇÃO DO PREENCHIMENTO ---
        grupo_reindex = grupo.set_index('num_ano').reindex(intervalo)
        mask_preenchidas = grupo_reindex['prodtonhl_v3'].isna()
        
        grupo_reindex['prodtonhl_v4'] = grupo_reindex['prodtonhl_v3'].interpolate(method='index', limit_direction='both')
        grupo_reindex.loc[mask_preenchidas, 'status_v08'] = status_preenchimento
        
        # --- [Correção] Adicionado 'flag_outlier' para não ser perdido
        static_cols = [c for c in grupo_reindex.columns if c not in ['prodtonhl_v1','prodtonhl_v2','prodtonhl_v3', 'prodtonhl_v4', 'status_v06', 'status_v07', 'status_v08', 'flag_outlier']]
        grupo_reindex[static_cols] = grupo_reindex[static_cols].ffill().bfill().infer_objects(copy=False)
        
        return grupo_reindex.reset_index()

    df_aptos_preenchimento = df_processado[df_processado['status_v06'] == 'Apto para Análise']
    if not df_aptos_preenchimento.empty:
        lista_grupos_preenchidos = []
        for _, grupo in df_aptos_preenchimento.groupby(group_cols):
            grupo_preenchido = _preencher_grupo(grupo, ano_min_geral, ano_max_geral)
            lista_grupos_preenchidos.append(grupo_preenchido)
        
        if lista_grupos_preenchidos:
            df_final_preenchido = pd.concat(lista_grupos_preenchidos, ignore_index=True)
            df_nao_processados = df_processado[df_processado['status_v06'] != 'Apto para Análise'].copy()
            df_processado = pd.concat([df_final_preenchido, df_nao_processados], ignore_index=True)

    # --- [RESUMO ETAPA 3] ---
    try:
        total_s3_saida = len(df_processado)
        mantidos_s3 = (df_processado['status_v08'] == 'Valor Mantido').sum()
        preenchidos_s3 = (df_processado['status_v08'].str.contains('Preenchido', na=False)).sum()
        zerados_s3 = (df_processado['status_v08'] == 'Zerado (Status Inválido/Outro)').sum()
        nao_aplicavel_s3 = (df_processado['status_v08'] == 'Não Aplicável').sum()
        
        print("\n--- Resumo Etapa 3 (Preenchimento) ---")
        print(f"Total de linhas na saída:    {total_s3_saida}")
        print(f"Linhas Originais Mantidas:   {mantidos_s3}")
        print(f"Linhas Novas (Preenchidas):  {preenchidos_s3}")
        print(f"Linhas Zeradas (Status):     {zerados_s3}")
        print(f"Linhas Não Aplicáveis:       {nao_aplicavel_s3} (Filtradas na Etapa 1)")
        print("--------------------------------------\n")
    except Exception as e:
        print(f"*** Erro ao gerar resumo S3: {e} ***")
    # --- [FIM RESUMO ETAPA 3] ---

    # --- FINALIZAÇÃO ---
    
    # --- [RESUMO GERAL - CORRIGIDO] ---
    try:
        total_final = len(df_processado)
        
        # 1. Encontra os dados ORIGINAIS que foram MANTIDOS
        mascara_mantidos = (
            (df_processado['status_v06'] == 'Apto para Análise') &
            (df_processado['status_v07'] == 'Valor Mantido') &
            (df_processado['status_v08'] == 'Valor Mantido')
        )
        qtd_mantidos = mascara_mantidos.sum()
        
        # 2. Pega a contagem de dados ADICIONADOS (da S3)
        qtd_preenchidos = (df_processado['status_v08'].str.contains('Preenchido', na=False)).sum()
        
        # 3. Calcula alterados com base no total INICIAL
        qtd_alterados_originais = total_inicial_consolidado - qtd_mantidos
        
        if total_inicial_consolidado > 0:
            porc_alteracao_original = (qtd_alterados_originais / total_inicial_consolidado) * 100
        else:
            porc_alteracao_original = 0.0
            
        print("\n--- Resumo Geral do Tratamento ---")
        print(f"Qtd Total de dados Iniciais (Consolidados): {total_inicial_consolidado}")
        print(f"  - Qtd Dados Mantidos (Originais):           {qtd_mantidos}")
        print(f"  - Qtd Dados Alterados/Filtrados (Originais):{qtd_alterados_originais}")
        print(f"  - Porcentagem de Alteração (sobre Iniciais):{porc_alteracao_original:.2f} %")
        print(f"Qtd Dados Adicionados (Preenchimento):      {qtd_preenchidos}")
        print(f"Qtd Total de dados (Saída Final):           {total_final}")
        print("-------------------------------------------\n")

    except Exception as e:
        print(f"*** Erro ao gerar resumo GERAL: {e} ***")
    # --- [FIM RESUMO GERAL] ---

    # A coluna 'flag_outlier' não é incluída intencionalmente na saída final
    colunas_finais = [c for c in df.columns if c not in ['prodtonhl_v1', 'flag_outlier']] + \
                     ['prodtonhl_v1', 'prodtonhl_v2', 'prodtonhl_v3', 'prodtonhl_v4', 
                      'status_v06', 'status_v07', 'status_v08']
    
    print("Tratamento de dados v4 concluído com sucesso!")
    # Garante que as colunas retornadas estejam na ordem correta
    colunas_existentes = [col for col in colunas_finais if col in df_processado.columns]
    
    return df_processado[colunas_existentes].sort_values(by=agg_cols).reset_index(drop=True)

### 6.2 Código

In [ ]:
#Como ajustei a classificação dos produtos posteriormente, mas o material
# foi feito de maneira manual, devo arrumar aqui

#Agrupo em grupos de alimentos emissores de COV para análises posteriores
df_producao_escala['tipo_industria_nfr'], df_producao_escala['food_color'] = zip(
    *df_producao_escala['Technology'].map(classificar_produto)
)

df_producao_final = tratamento_outliers_v3(df_producao_escala)

df_inventario = df_producao_final.copy()
df_inventario['Emissão NMCOV (kg)'] = (df_inventario['prodtonhl_v4'] * df_inventario['Value'].astype(float))
df_inventario['Emissão NMCOV CI_lower (kg)'] = (df_inventario['prodtonhl_v4'] * df_inventario['CI_lower'].astype(float))
df_inventario['Emissão NMCOV CI_upper (kg)'] = (df_inventario['prodtonhl_v4'] * df_inventario['CI_upper'].astype(float))

# Como edgar é em ton, vou deixar tudo em toneladas
df_inventario['Emissão NMCOV (ton)'] = df_inventario['Emissão NMCOV (kg)']/1000
df_inventario['Emissão NMCOV CI_lower (ton)'] = df_inventario['Emissão NMCOV CI_lower (kg)']/1000
df_inventario['Emissão NMCOV CI_upper (ton)'] = df_inventario['Emissão NMCOV CI_upper (kg)']/1000

## 7. Exportar para realizar análises em outro código

In [ ]:
df_inventario['CNPJ'] = "'" + df_inventario['CNPJ'].astype(str)
df_inventario = df_inventario[df_inventario['prodtonhl_v4'] != 0].copy()

df_inventario.to_csv(os.path.join(repo_path,'outputs','inventarioEmissoesIndustriaisIndustriaAlimenticiaBR_V3.csv'), index = False, encoding='latin1')